In [5]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# === CONFIG ===
subject_csv = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\subject_level_cca_fixedlagTHESISMemhold2.csv")
weights_dir = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\weights_Memhold2")

# === Load subject-level results and keep only memory condition ===
df = pd.read_csv(subject_csv)
df_mem = df[df["condition"].str.lower() == "memory"].copy()

required_cols = {"subject", "condition", "lag_ms", "r"}
missing = required_cols - set(df_mem.columns)
if missing:
    raise ValueError(f"Missing columns in subject CSV: {missing}")

def sign(x: float):
    if x > 0: return 1
    if x < 0: return -1
    return 0

def weights_path_for(subject: int, lag_ms: int) -> Path:
    sub_str = f"sub-{int(subject):03d}"
    lag_str = f"+{lag_ms}ms" if lag_ms > 0 else f"{lag_ms}ms"
    return weights_dir / f"{sub_str}_eeg_weights_mem_{lag_str}.csv"

# Tallies per channel
match_counts = {}                # channel -> # matches
mismatch_counts = {}             # channel -> # mismatches
appearance_counts = {}           # channel -> # nonzero weights seen

sum_abs_r_match = {}             # channel -> sum of |r| when signs match
sum_abs_r_mismatch = {}          # channel -> sum of |r| when signs DO NOT match

per_subject_rows = []            # detailed log

for _, row in df_mem.iterrows():
    subj = int(row["subject"])
    lag  = int(row["lag_ms"])
    r    = float(row["r"])
    sr   = sign(r)

    if sr == 0:
        per_subject_rows.append({
            "subject": subj, "lag_ms": lag, "r": r, "r_sign": 0,
            "weights_file": None, "status": "skip: r == 0"
        })
        continue

    wpath = weights_path_for(subj, lag)
    if not wpath.exists():
        per_subject_rows.append({
            "subject": subj, "lag_ms": lag, "r": r, "r_sign": sr,
            "weights_file": str(wpath), "status": "missing weights file"
        })
        continue

    try:
        wdf = pd.read_csv(wpath)
    except Exception as e:
        per_subject_rows.append({
            "subject": subj, "lag_ms": lag, "r": r, "r_sign": sr,
            "weights_file": str(wpath), "status": f"failed to read weights: {e}"
        })
        continue

    if not {"channel", "weight"}.issubset(wdf.columns):
        per_subject_rows.append({
            "subject": subj, "lag_ms": lag, "r": r, "r_sign": sr,
            "weights_file": str(wpath), "status": "weights file missing required columns"
        })
        continue

    abs_r = abs(r)

    for _, wrow in wdf.iterrows():
        ch = str(wrow["channel"])
        wt = float(wrow["weight"])
        sw = sign(wt)

        if sw == 0:
            # skip zero-weight channels from both appearance and rate
            continue

        # track that this channel appeared (with non-zero weight) for this subject
        appearance_counts[ch] = appearance_counts.get(ch, 0) + 1

        if sw == sr:
            match_counts[ch] = match_counts.get(ch, 0) + 1
            sum_abs_r_match[ch] = sum_abs_r_match.get(ch, 0.0) + abs_r
        else:
            mismatch_counts[ch] = mismatch_counts.get(ch, 0) + 1
            sum_abs_r_mismatch[ch] = sum_abs_r_mismatch.get(ch, 0.0) + abs_r

    per_subject_rows.append({
        "subject": subj, "lag_ms": lag, "r": r, "r_sign": sr,
        "weights_file": str(wpath), "status": "ok"
    })

# Build summary
channels = sorted(set(appearance_counts) | set(match_counts) | set(mismatch_counts))

def safe_div(num, den):
    return (num / den) if den and den != 0 else np.nan

summary = pd.DataFrame({
    "channel": channels,
    "matches": [match_counts.get(ch, 0) for ch in channels],
    "mismatches": [mismatch_counts.get(ch, 0) for ch in channels],
    "appearances_nonzero": [appearance_counts.get(ch, 0) for ch in channels],
    "mean_abs_r_match": [
        safe_div(sum_abs_r_match.get(ch, 0.0), match_counts.get(ch, 0)) for ch in channels
    ],
    "mean_abs_r_mismatch": [
        safe_div(sum_abs_r_mismatch.get(ch, 0.0), mismatch_counts.get(ch, 0)) for ch in channels
    ],
    "mean_abs_r_diff": [
        safe_div(sum_abs_r_match.get(ch, 0.0), match_counts.get(ch, 0)) - safe_div(sum_abs_r_mismatch.get(ch, 0.0), mismatch_counts.get(ch, 0)) for ch in channels
    ]
})

summary["match_rate"] = summary.apply(
    lambda r: safe_div(r["matches"], r["appearances_nonzero"]), axis=1
)

summary = summary.sort_values(
    ["matches", "match_rate", "channel"],
    ascending=[False, False, True]
)

# Save
out_summary = subject_csv.parent / "sign_match_summary_by_channel.csv"
out_log     = subject_csv.parent / "sign_match_per_subject_log.csv"
summary.to_csv(out_summary, index=False)
pd.DataFrame(per_subject_rows).to_csv(out_log, index=False)

# Print a compact view
cols_to_show = [
    "channel", "matches", "mismatches", "appearances_nonzero",
    "match_rate", "mean_abs_r_match", "mean_abs_r_mismatch", "mean_abs_r_diff"
]
print("Top electrodes by sign-match count (ties broken by match_rate):")
print(summary[cols_to_show].head(20).to_string(index=False))
print(f"\nSaved summary: {out_summary}")
print(f"Saved per-subject log: {out_log}")


Top electrodes by sign-match count (ties broken by match_rate):
channel  matches  mismatches  appearances_nonzero  match_rate  mean_abs_r_match  mean_abs_r_mismatch  mean_abs_r_diff
     C2       33          21                   54    0.611111          0.047733             0.057509        -0.009776
     C1       31          23                   54    0.574074          0.055134             0.046684         0.008450
    FC3       31          23                   54    0.574074          0.052659             0.050020         0.002639
     F1       30          24                   54    0.555556          0.055115             0.047060         0.008056
     F4       30          24                   54    0.555556          0.045507             0.059070        -0.013563
    FC4       30          24                   54    0.555556          0.057541             0.044028         0.013513
    AFz       29          25                   54    0.537037          0.051027             0.052125        -0